In [0]:
from datetime import datetime

In [0]:
dbutils.widgets.text("batch_id" , "1" , "Batch Id (1,2,or 3)")


In [0]:
## read from the widgets 
batch_id = dbutils.widgets.get("batch_id").strip()
run_id = datetime.now().strftime("%Y%m%d%H%M%S")

print(f"Run_id is {run_id}")
print(f"Batch_id is {batch_id}")

In [0]:
team_name = "team_lemma"
source_base = "abfss://raw@schwabdldevsa.dfs.core.windows.net"
catalog     = f"charles_schwab_retailbrokerage_dev_{team_name}"
landing_volume = f"/Volumes/{catalog}/landing/pwg"

Batch_Folder = f"Batch{batch_id}"
source_folder = f"{source_base}/{Batch_Folder}"
target_path = f"{landing_volume}/{Batch_Folder}/finwire"

print(target_path)

In [0]:
## finwire has only one file present in the batch 



if batch_id !="1":
    
    dbutils.notebook.exit(F"Finewire file not present in the {batch_id}")

In [0]:
all_files = dbutils.fs.ls(source_folder)

finwire_paths = [
    f.path for f in all_files 
    if f.name.startswith("FINWIRE")
    and "_audit" not in f.name
    and not f.name.endswith(".csv")
]

print(f"found{len(finwire_paths)} FINWIRE files in {Batch_Folder}")
assert len(finwire_paths) == 203 , f"EXpected 203 FINWIRE files in {Batch_Folder} found {len(finwire_paths)}"


### read all finwire files as raw text

In [0]:
from pyspark.sql.functions import lit , current_timestamp

In [0]:
df_raw = spark.read.text(finwire_paths)


print(f"Found {df_raw.count()}")

In [0]:
### adding metadata columns

df_landing = df_raw\
                .withColumnRenamed("value" , "raw_line")\
                .withColumn("_source_file" , lit("FINWIRE"))\
                .withColumn("_batch" , lit(batch_id))\
                .withColumn("_landing_ts" , current_timestamp())\
                .withColumn("_run_id" , lit(run_id))

print("landing finwire schema ")
df_landing.printSchema()

In [0]:
## write the files in the parquet format

df_landing\
    .write\
    .mode("overwrite")\
    .parquet(target_path)

In [0]:
landing_count = spark.read.parquet(target_path).count()
source_count = df_landing.count()

print(f"source count : {source_count}")
print(f"landing count : {landing_count}")
if source_count==landing_count:
    print("Match")
else:
    print("Not MAtch")

### logging cell

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql import Row

recon_results = [
    Row(
        source_table = "finwire",
        batch_id = Batch_Folder,
        target_path = target_path,
        source_count = source_count,
        landing_count = landing_count,
        status = "Match" if source_count==landing_count else "Not Match",
        columns_applied = "raw_line"
    )
]

recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if row.status in ("Match" , "Not Match"):
        log_pipeline_recon(
            spark         = spark,
            run_id        = run_id,
            batch_id      = row.batch_id,
            domain        = "MARKET",
            table_name    = row.source_table,
            source_layer  = "raw",
            target_layer  = "landing",
            source_count  = row.source_count,
            target_count  = row.landing_count
        )

        log_audit_event(
            spark = spark,
            run_id = run_id,
            batch = row.batch_id,
            layer = "Landing",
            table_name = row.source_table,
            operation = "Append",
            rows_affected  = row.landing_count
        )

display(recon_df)

In [0]:
# landing_base = f"/Volumes/charles_schwab_retailbrokerage_dev_{team_name}/landing/pwg/Batch1/"

# items = dbutils.fs.ls(landing_base)

# for item in items:
#     if item.name.upper().startswith("FINWIRE"):
#         dbutils.fs.rm(item.path, True)
#         print(f"Deleted: {item.path}")

# print("Cleanup complete!")